# 🛡️ Real-Time Fraud Detection & Risk Intelligence
## Notebook 02: Model Benchmarking, Threshold Optimization & SHAP Explainability

This notebook demonstrates:
1. Stratified Train / Validation / Test data splitting preventing leakage
2. Benchmarking Logistic Regression, Random Forest, and XGBoost
3. Evaluating Precision-Recall Curves (PR-AUC) and ROC-AUC
4. Optimal Threshold Search on validation split to maximize F1
5. SHAP feature attribution waterfall plots

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
from src.data.ingestion import DataIngestion
from src.data.preprocessor import DataPreprocessor
from src.models.trainer import ModelTrainer
from src.models.evaluator import ModelEvaluator
from src.models.threshold_optimizer import ThresholdOptimizer
from src.models.explainability import FraudExplainer
%matplotlib inline

### 1. Leakage-Free Preprocessing

In [ ]:
ingestion = DataIngestion()
df = ingestion.load_data()
preprocessor = DataPreprocessor()
X_train_df, X_val_df, X_test_df, y_train, y_val, y_test = preprocessor.split_data(df)
X_train, X_val, X_test = preprocessor.fit_transform(X_train_df, X_val_df, X_test_df)
print(f'Transformed Train Shape: {X_train.shape}')

### 2. Multi-Model Benchmark

In [ ]:
trainer = ModelTrainer()
lr = trainer.train_logistic_regression(X_train, y_train.values)
rf = trainer.train_random_forest(X_train, y_train.values)
xgb = trainer.train_xgboost(X_train, y_train.values)

models = {'Logistic Regression': lr, 'Random Forest': rf, 'XGBoost': xgb}
val_metrics = {name: ModelEvaluator.evaluate(m, X_val, y_val.values) for name, m in models.items()}
display(ModelEvaluator.compare_models(val_metrics))

### 3. Threshold Optimization for Best Model

In [ ]:
best_model = xgb
opt_result = ThresholdOptimizer.find_optimal_threshold(best_model, X_val, y_val.values)
print(f'Optimal Threshold: {opt_result["optimal_threshold"]}')
print(f'Optimal F1-Score: {opt_result["best_f_score"]}')

df_curve = pd.DataFrame(opt_result['curve_data'])
plt.figure(figsize=(8, 4))
plt.plot(df_curve['threshold'], df_curve['precision'], label='Precision')
plt.plot(df_curve['threshold'], df_curve['recall'], label='Recall')
plt.plot(df_curve['threshold'], df_curve['f_beta'], label='F1-Score', linewidth=2)
plt.axvline(opt_result['optimal_threshold'], color='r', linestyle='--', label=f'Optimal τ* = {opt_result["optimal_threshold"]}')
plt.title('Threshold Optimization Trade-off Curve')
plt.xlabel('Decision Threshold')
plt.ylabel('Score')
plt.legend()
plt.show()